In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
import random
import tensorflow as tf
import cv2

from src.font import FONT_PATHS, FONT_TO_LABEL, LABEL_TO_FONT_NAME
from src.parse_sudoku_board import preprocess_digit

In [3]:
# Get the project directory
PROJECT_DIR = Path.cwd().parents[0]

In [4]:
# Sizes
IMG_SIZE = 64
MODEL_SIZE = 64
BATCH_SIZE = 32

In [5]:
# Digits from 1 to 9
DIGITS = list(range(1, 10))

In [6]:
# The paths of the fonts
NUM_CLASSES = len(FONT_TO_LABEL)

In [7]:
def scale_font_to_target(draw: ImageDraw.Draw, font_path: str, text: str, canvas_size: int, target_scale: float = 0.8) -> ImageFont.FreeTypeFont:
    """
    Finds the largest font size that fits the text within a canvas with given dimension.

    Args:
        draw: The 2D drawing interface
        font_path: The file path of the font
        text: The text to put on the canvas
        img_size: The size of the canvas
        target_scale: The desired scale (default to 0.8)

    Returns:
        ImageFont.FreeTypeFont: The largest font size
    """

    target = canvas_size * target_scale

    low, high = 10, 300
    best_font = ImageFont.truetype(font_path, low)

    # Use binary search
    # The search range is [low, high]
    while low <= high:
        mid = (low + high) // 2
        font = ImageFont.truetype(font_path, mid)

        left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
        w, h = right - left, bottom - top

        if max(w, h) <= target:
            best_font = font
            low = mid + 1
        else:
            high = mid - 1

    return best_font

In [8]:
def render_digit(digit: int, font_path: str) -> np.ndarray:
    """
    Renders a digit on the center of an image.

    Args:
        digit: The digit to put on the canvas
        font_path: The file path to the font

    Returns:
        np.ndarray: An image with a digit at the center
    """

    # Create a white image
    img = Image.new("L", (IMG_SIZE, IMG_SIZE), 255)

    draw = ImageDraw.Draw(img)

    font = scale_font_to_target(
        draw,
        font_path,
        str(digit),
        canvas_size=IMG_SIZE,
        target_scale=random.uniform(0.65, 0.9)
    )

    # Find the width and height of the text bounding box
    left, top, right, bottom = draw.textbbox(
        (0, 0),
        str(digit),
        font=font
    )

    # Find the location to draw the digit
    w = right - left
    h = bottom - top
    x = (IMG_SIZE - w) // 2 + random.randint(-10, 10)
    y = (IMG_SIZE - h) // 2 + random.randint(-10, 10)

    draw.text(
        (x, y),
        str(digit),
        fill=0,
        font=font
    )

    # Rotate the image with a random angle and fill the empty spot with white pixels
    angle = random.uniform(-10, 10)
    img = img.rotate(angle, fillcolor=255)

    return np.array(img)

In [9]:
def apply_camera_effects(img: np.ndarray) -> np.ndarray:
    """
    Applies random camera degradation effects.

    Args:
        img: The input image

    Retruns:
        np.ndarray: The degraded image
    """

    img = img.astype(np.uint8)

    # Blurriness
    if random.random() < 0.7:

        sigma = random.uniform(0.3, 2.0)

        img = cv2.GaussianBlur(img, (0, 0), sigma)

    # Contrast & brightness
    if random.random() < 0.5:

        # Contrast (α)
        alpha = random.uniform(0.7,1.3)

        # Brightness (β)
        beta = random.uniform(-40, 40)

        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    # Noises
    if random.random() < 0.5:

        # Cast the image to float since we are going to apply noises
        img_float = img.astype(np.float32)

        # Generate noises
        noise = np.random.normal(0, random.uniform(5, 20), img.shape)

        # Add the noises to the input image
        img_float += noise

        # Cast the image back to uint
        img = np.clip(img_float, 0, 255).astype(np.uint8)

    # JPEG artifacts
    if random.random() < 0.5:

        quality = random.randint(25, 90)

        _, encoded = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])

        img = cv2.imdecode(encoded, cv2.IMREAD_GRAYSCALE)

    return img.astype(np.uint8)

In [10]:
def generate_digit_sample():

    while True:

        digit = random.randint(1, 9)
        font_path = random.choice(FONT_PATHS)
        label = FONT_TO_LABEL[font_path]

        img = render_digit(digit, font_path)
        img = apply_camera_effects(img)

        # Blur the image
        blur = cv2.GaussianBlur(img, (3, 3), 0)

        # Use adaptive threshold
        # NOTE: block size should be an odd number
        thresh = cv2.adaptiveThreshold(
            blur,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,
            blockSize=11,
            C=2
        )

        # Remove noises by using a 2x2 kernel
        kernel = np.ones((2, 2), np.uint8)
        thresh = cv2.morphologyEx(
            thresh,
            cv2.MORPH_OPEN,
            kernel
        )

        # Find the contour of the digit
        contours, _ = cv2.findContours(
            thresh,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if not contours:
            continue

        # Find the largest area
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)

        # If the largest area is less than a threshold, then it must be a noise or an artifact
        cell_area = img.shape[0] * img.shape[1]
        min_area_threshold = cell_area * 0.01
        if area < min_area_threshold:
            continue

        # Find the bounding box that bounds the digit
        x, y, w, h = cv2.boundingRect(largest)

        # If the bounding box is small that means there is no digit to crop
        if w == 0 or h == 0:
            continue

        # Crop the digit
        digit = thresh[y:y+h, x:x+w]

        processed = preprocess_digit(digit, int(IMG_SIZE * 0.8), IMG_SIZE)

        # Convert to (IMG_SIZE, IMG_SIZE, 1)
        sample = np.expand_dims(processed, axis=-1)

        return sample.astype(np.float32), label

In [11]:
def data_generator():
    """
    Data generator wrapper that yields an RGB image and its label

    Yields:
        tuple:
            np.ndarray: An RGB image that contains an augmented digit
            int: The label of the image (zero-based index)
    """

    while True:
        yield generate_digit_sample()

In [12]:
# Describe what the custom generator will yield
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32), # the image
    tf.TensorSpec(shape=(), dtype=tf.int32) # the label
)

# Create the training dataset from the generator
train_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

# Use prefetching to improve performance
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

2026-06-07 18:16:08.312157: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-06-07 18:16:08.312178: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-06-07 18:16:08.312181: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1780874168.312198 1441746 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1780874168.312218 1441746 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [13]:
# Create the training dataset from the generator
val_ds = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=output_signature
)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [14]:
# Create a model
model = tf.keras.Sequential([

    # First layer
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(MODEL_SIZE, MODEL_SIZE, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Second layer
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Third layer
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Forth layer
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    # Final dense layer for classification
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Summary of the model
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 29, 29, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 4, 4, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 4, 4, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 2, 2, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 697,797 (2.66 MB)

 Trainable params: 696,837 (2.66 MB)

 Non-trainable params: 960 (3.75 KB)

In [16]:
x, y = next(iter(train_ds))
print(x.shape)

(32, 64, 64, 1)


In [17]:
# Create a callback to stop training when the improvement metric has stopped improving
callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', # set the improvement metric to value loss
    patience=3, # the number of epochs to wait
    restore_best_weights=True   # roll back the model to its best performance state
)

In [18]:
history = model.fit(
    train_ds,
    steps_per_epoch=250,
    epochs=50,
    callbacks=[callback]
)

Epoch 1/50


2026-06-07 18:16:09.080021: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.2881 - loss: 6.9136
Epoch 2/50
  5/250 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.2870 - loss: 6.6952

/Users/neil/Develop/SudoVisionCruncher/.venv/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.3487 - loss: 6.6479
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.4304 - loss: 5.7418
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.5222 - loss: 4.8019
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.6184 - loss: 3.8640
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 36ms/step - accuracy: 0.6745 - loss: 3.3788
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.7266 - loss: 2.7685
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.7523 - loss: 2.5538
Epoch 9/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.7747 - loss: 2.2844
Epoch 10/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.8055 - loss: 1.8629
Epoch 11/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.8207 - loss: 1.7820
Epoch 12/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/step - accuracy: 0.8267 - loss: 1.6944
Epoch 13/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 37ms/st

In [19]:
# Evaluate the model
test_loss, test_acc = model.evaluate(val_ds, steps=5, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

5/5 - 0s - 60ms/step - accuracy: 0.9438 - loss: 0.5128

Test Accuracy: 94.38%


In [20]:
# Save the model and its parameters
model.save(f'{PROJECT_DIR}/models/font_recognition_MobileNetV2.keras')